# Tutorial: Using the blendemu Emulators

This notebook walks you through how to apply a pre-trained `blendemu` emulator suite to your own galaxy catalogue.

### What are the emulators?

`blendemu` provides a trio of trained XGBoost models per survey:

1. **Classification emulator** — predicts the probability that a galaxy will be *detected* by SExtractor in a survey image. Takes the galaxy's own properties + its nearest neighbor's properties, outputs `p(detected)`.
2. **Blending-response regression emulator** — predicts $\delta e_t / \gamma$: how much a *primary* galaxy's measured shape is perturbed by the shear of a nearby neighbor (blending contamination), per unit applied shear.
3. **Self-response regression emulator** — predicts $\delta e_t / \gamma$ in response to the *galaxy's own* shear: essentially `(1 + m)` where m is the per-galaxy multiplicative bias. Captures noise bias / shape-prior dilution for the target galaxy itself.

Combined, they let you correct the effective source redshift distribution $n_\gamma(z)$ used in weak lensing analyses.

### What you'll do here

1. Load a pre-trained emulator suite (tagged `lsst_r` here).
2. Inspect what each emulator expects as input.
3. Apply the **classification** emulator → detection probabilities.
4. Apply the **blending-response** emulator → per-pair shear response.
5. Apply the **self-response** emulator → per-galaxy self shear response.
6. Combine them for the n(z) correction (one-liner via `BlendingPredictor.correct_nz`).

See also [inference_nz_correction.ipynb](inference_nz_correction.ipynb) for a fuller n(z)-correction example including HSC tomographic bins.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import os, sys

# Add blendemu to path (only needed if not installed)
sys.path.insert(0, '..')
from blendemu import BlendingPredictor
from blendemu import data_utils, nz_utils

plt.rcParams.update({
    'font.family': 'serif', 'mathtext.fontset': 'cm',
    'figure.dpi': 120,
})

## 2. Loading the Emulators

The `BlendingPredictor.load()` classmethod loads both models + their standardization and feature boundaries in one call. You need to supply:

- `model_dir` — where the model files live
- `tag` — the survey/data tag used when training (e.g. `'lsst_r'`)
- `conditions` — the observing conditions used during training (pixel scale, PSF FWHM, noise RMS, etc.). These are used to rescale your catalogue features so they match the training distribution.

In [ ]:
# The observing conditions must match what the emulators were TRAINED on.
# For the LSST r-band models: RMS=0.312, PSF FWHM=0.73", pixel scale=0.2"/pix
LSST_R_CONDITIONS = {
    'pixel_size': 0.2,
    'zero_point': 30,
    'psf_fwhm': 0.73,
    'moffat_beta': 2.224,
    'pixel_rms': 0.312,
}

predictor = BlendingPredictor.load(
    model_dir='./models',
    tag='lsst_r',
    conditions=LSST_R_CONDITIONS,
)

print('Emulators loaded.')
print(f'  Classification features:   {predictor.bst_cla.feature_names}')
print(f'  Blending-response features: {predictor.bst_reg.feature_names}')
if predictor.bst_self is not None:
    print(f'  Self-response features:    {predictor.bst_self.feature_names}')
print(f'  Blending y standardization: mean={predictor.y_mean:.4f}, std={predictor.y_std:.4f}')
if predictor.y_mean_self is not None:
    print(f'  Self     y standardization: mean={predictor.y_mean_self:.4f}, std={predictor.y_std_self:.4f}')

## 3. Your Input Catalogue

You need a galaxy catalogue with the following columns:

| Column | Description |
|--------|-------------|
| `RA`, `DEC` | Sky positions (degrees) |
| `redshift` | Redshift |
| `r` | r-band magnitude |
| `Re` | Half-light radius (arcsec) |
| `sersic_n` | Sérsic index |
| `axis_ratio` | Axis ratio b/a |

These are the *input* (true) properties — the emulators handle the detection and shape-measurement effects internally.

In [ ]:
# Load the bundled example catalogue
icat = pd.read_feather('../data/example_catalog.feather')
print(f'Catalogue shape: {icat.shape}')
print(f'Columns: {list(icat.columns)}')
icat.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].hist(icat['r'], bins=50, histtype='step', linewidth=2)
axes[0].set_xlabel('r-band mag'); axes[0].set_ylabel('Count')
axes[1].hist(icat['redshift'], bins=50, histtype='step', linewidth=2)
axes[1].set_xlabel('redshift')
axes[2].hist(icat['Re'], bins=50, histtype='step', linewidth=2)
axes[2].set_xlabel(r'$R_e$ [arcsec]')
plt.tight_layout()

## 4. Predict Detection Probability

The classification emulator answers: *given a galaxy and its nearest neighbor, what's the probability this galaxy gets detected by SExtractor?*

Call `predictor.predict_detection(icat)` — it returns a `(detect_prob, feature_df)` tuple.

In [ ]:
cla_features = predictor.predict_detection(icat)
detect_prob = cla_features['detection_prob'].values
print(f'Detection probabilities for {len(detect_prob):,} galaxies')
print(f'  mean:      {detect_prob.mean():.3f}')
print(f'  fraction with p > 0.5: {(detect_prob > 0.5).mean():.3f}')
print(f'  fraction with p > 0.9: {(detect_prob > 0.9).mean():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(detect_prob, bins=50, histtype='step', linewidth=2)
axes[0].set_xlabel('Predicted $p$(detected)'); axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of detection probabilities')

# Detection rate vs magnitude (classic survey-depth curve)
mag_bins = np.linspace(icat['r'].min(), icat['r'].max(), 30)
mag_c = 0.5 * (mag_bins[:-1] + mag_bins[1:])
p_by_mag = [detect_prob[(icat['r'] >= lo) & (icat['r'] < hi)].mean()
            for lo, hi in zip(mag_bins[:-1], mag_bins[1:])]
axes[1].plot(mag_c, p_by_mag, 'o-', linewidth=2)
axes[1].set_xlabel('r-band mag'); axes[1].set_ylabel('Mean predicted $p$(detected)')
axes[1].set_ylim(0, 1.05); axes[1].set_title('Detection probability vs magnitude')
plt.tight_layout()

## 5. Predict Blending Response

The blending-response regression emulator answers: *for a primary galaxy with a given sheared neighbor, what's the per-pair blending contamination $\delta e_t / \gamma$?*

It takes **two** catalogues: the set of primaries (galaxies whose shape you care about) and the set of potential secondaries (neighbors that could blend with them). The helper `predict_response` handles the KDTree pairing, feature rescaling, and inverse standardization.

The mean signal is small (per-pair values ~0.001-0.01 for typical separations) since blending is a subtle contamination effect.

In [ ]:
# For demo: use the whole catalogue as both primaries and potential neighbors
reg_features = predictor.predict_response(icat_pri=icat, icat_sec=icat)

print(f'Primary-neighbor pairs found: {reg_features.shape[0]:,}')
print(f'Mean response <delta_et/gamma>: {reg_features["response"].mean():.5f}')
print(f'Predicted columns: {[c for c in reg_features.columns if "response" in c or "scaled" in c][:5]}...')

In [ ]:
# Response vs nearest-neighbor distance — closer neighbors drive larger blending response
dist_bins = np.linspace(0, reg_features['distance'].quantile(0.99), 25)
dist_c = 0.5 * (dist_bins[:-1] + dist_bins[1:])
resp_by_dist = [reg_features['response'][(reg_features['distance'] >= lo) & (reg_features['distance'] < hi)].mean()
                for lo, hi in zip(dist_bins[:-1], dist_bins[1:])]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dist_c, resp_by_dist, 'o-', linewidth=2, color='C3')
ax.axhline(0, color='k', ls=':', alpha=0.5)
ax.set_xlabel('Primary-neighbor distance [arcsec]')
ax.set_ylabel(r'Predicted $\delta e_t / \gamma$')
ax.set_title('Blending response falls off with separation')
plt.tight_layout()

## 6. Predict Self-Response

The self-response regression emulator answers: *for a given galaxy, what's $\delta e_t / \gamma$ in response to the galaxy's **own** shear?*

This is the classical shear response — essentially `(1 + m)` where `m` is the per-galaxy multiplicative bias (usually negative due to noise bias / shape-prior dilution, especially for faint galaxies).

API mirrors `predict_response`: pass a primaries catalogue and a secondaries catalogue (used for nearest-neighbor features). The output column is named `'self_response'` to distinguish it from blending.

In [ ]:
self_features = predictor.predict_self_response(icat_pri=icat, icat_sec=icat)

print(f'Target-neighbor pairs evaluated: {self_features.shape[0]:,}')
print(f'Mean self-response <delta_et/gamma>: {self_features["self_response"].mean():.4f}')
print(f'  (Should be ~1 for well-resolved, bright galaxies;')
print(f'   drops below 1 for faint/small galaxies due to noise bias.)')
print(f'Predicted column: "self_response" (in units of 1 per applied shear)')

In [ ]:
# Self-response vs magnitude — expect ~1 for bright galaxies, diluted for faint ones
mag_bins = np.linspace(18, 27, 25)
mag_c = 0.5 * (mag_bins[:-1] + mag_bins[1:])
self_r = self_features['self_response'].values
mag_vals = self_features['r_input_p'].values

self_by_mag = [self_r[(mag_vals >= lo) & (mag_vals < hi)].mean()
               for lo, hi in zip(mag_bins[:-1], mag_bins[1:])]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(self_r, bins=100, histtype='step', linewidth=2, range=(-1, 3))
axes[0].axvline(1, color='k', ls='--', alpha=0.5, label='unbiased = 1')
axes[0].axvline(0, color='k', ls=':', alpha=0.3)
axes[0].set_xlabel(r'Predicted $\delta e_t / \gamma$ (self-response)')
axes[0].set_ylabel('Count'); axes[0].legend()
axes[0].set_title('Self-response distribution')

axes[1].plot(mag_c, self_by_mag, 'o-', linewidth=2, color='C2')
axes[1].axhline(1, color='k', ls='--', alpha=0.5, label='unbiased')
axes[1].set_xlabel('r-band mag'); axes[1].set_ylabel('Mean self-response')
axes[1].set_title('Multiplicative bias trend vs magnitude')
axes[1].legend()

plt.tight_layout()

## 7. The Full n(z) Correction (one-liner)

`predictor.correct_nz(icat, (dndz, z))` wraps everything: resamples the catalogue to match your target n(z), predicts detection probabilities and blending-response values, and integrates to compute $\Delta n_\gamma(z)$.

(Currently the n(z) correction uses the blending-response emulator only; self-response would modify the per-galaxy shear weight but the pipeline factorizes this differently.)

In [ ]:
# Make up a target Gaussian n(z) around z=0.7
z_grid = np.linspace(0, 2, 201)
dndz = np.exp(-(z_grid - 0.7)**2 / (2 * 0.1**2))
dndz /= dndz.sum() * (z_grid[1] - z_grid[0])   # normalize

reg_cat, z, delta_n = predictor.correct_nz(icat, (dndz, z_grid))
new_nz = dndz + delta_n

# Compare mean redshifts
old_mean_z = np.trapz(dndz * z, z)
new_mean_z = np.trapz(new_nz * z, z)
print(f'Original <z>: {old_mean_z:.4f}')
print(f'Corrected <z>: {new_mean_z:.4f}')
print(f'Shift: {new_mean_z - old_mean_z:+.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(z_grid, dndz, linewidth=2, label=r'original $n(z)$')
axes[0].plot(z, new_nz, linewidth=2, linestyle='--', label=r'blending-corrected $n(z)$')
axes[0].set_xlabel('redshift'); axes[0].legend()

axes[1].plot(z, delta_n, linewidth=2, color='C3')
axes[1].axhline(0, color='k', ls=':', alpha=0.5)
axes[1].set_xlabel('redshift'); axes[1].set_ylabel(r'$\Delta n_\gamma(z)$')
axes[1].set_title('Blending-induced redshift mixing correction')
plt.tight_layout()

## 8. Applying to Your Own Data

Minimum recipe:

```python
from blendemu import BlendingPredictor
import pandas as pd, numpy as np

predictor = BlendingPredictor.load(
    model_dir='/path/to/blendemu/models',
    tag='lsst_r',           # or whatever tag matches your training
    conditions={...},       # match the training observing conditions
)

icat = pd.read_feather('my_catalogue.feather')

# Individual predictions
detect_prob, _ = predictor.predict_detection(icat)
blend_pairs   = predictor.predict_response(icat_pri=icat, icat_sec=icat)
self_pairs    = predictor.predict_self_response(icat_pri=icat, icat_sec=icat)

# Or the full n(z) correction in one call
_, z, delta_n = predictor.correct_nz(icat, (my_dndz, my_z))
corrected_nz = my_dndz + delta_n
```

### Required catalogue columns
`RA`, `DEC`, `redshift`, `r`, `Re`, `sersic_n`, `axis_ratio`

### Choosing the right emulator tag
Models are trained per-survey because detection and shape-measurement bias depend on noise, PSF, and pixel scale. Use the tag matching your target survey:
- `lsst_r` — LSST r-band, 10-year depth
- (others as you train them)

If you're applying to a *different* survey, retrain using that survey's noise parameters (see `configs/` and `scripts/train_emulator.py`).

### CLI alternative
```bash
python scripts/run_inference.py \
    --config configs/fs2_lsst_r.yaml \
    --catalogue my_catalogue.feather \
    --nz-file hsc_nz.fits \
    --output corrected_nz.fits
```

## 9. Advanced: Predicting on Custom / Pre-Paired Data

The methods above (`predict_detection`, `predict_response`, `predict_self_response`) all expect **full catalogues** and do their own KDTree neighbor-finding internally.

If you already have a paired catalogue (e.g. from your own detection + matching code), or you just want the prediction for a single galaxy pair, use these lower-level methods instead:

- `predict_on_pairs(pair_df, task=...)` — takes a DataFrame with pair-level raw features and returns predictions
- `predict_one_pair(primary={...}, neighbor={...}, task=...)` — convenience wrapper for a single pair

### Example A: A single galaxy (or single pair)

Quick sanity checks or what-if experiments — e.g. "what does the emulator predict for a galaxy with $R_e = 0.4''$, r = 23.5, and a close neighbor?"

In [ ]:
# Detection probability for an isolated galaxy (no neighbor)
p_det = predictor.predict_one_pair(
    primary={'Re': 0.4, 'r': 23.5, 'sersic_n': 1.2},
    task='detection',
)
print(f'p(detected) for isolated galaxy r=23.5: {p_det:.3f}')

# Blending response: same primary, with a close sheared neighbor at 2 arcsec
response = predictor.predict_one_pair(
    primary={'Re': 0.4, 'r': 23.5, 'sersic_n': 1.2},
    neighbor={'Re': 0.3, 'r': 24.5, 'sersic_n': 2.0, 'distance': 2.0},
    task='response',
)
print(f'Blending response at d=2": {response:.4f}')

# Self-response for the same galaxy
self_r = predictor.predict_one_pair(
    primary={'Re': 0.4, 'r': 23.5, 'sersic_n': 1.2},
    neighbor={'Re': 0.3, 'r': 24.5, 'sersic_n': 2.0, 'distance': 2.0},
    task='self_response',
)
print(f'Self-response: {self_r:.4f}  (1.0 = unbiased, <1 = noise-biased)')

### Example B: A pre-paired DataFrame

Your pair DataFrame needs these raw-feature columns:
`Re_input_p`, `r_input_p`, `sersic_n_input_p` (target galaxy), `Re_input_s`, `r_input_s`, `sersic_n_input_s` (neighbor), `distance` (arcsec).

`predict_on_pairs` handles the rescaling + inverse standardization internally.

In [ ]:
# Example: a small custom pair catalogue built by the user
custom_pairs = pd.DataFrame([
    {'Re_input_p': 0.4, 'r_input_p': 23.5, 'sersic_n_input_p': 1.2,
     'Re_input_s': 0.3, 'r_input_s': 24.5, 'sersic_n_input_s': 2.0, 'distance': 2.0},
    {'Re_input_p': 0.6, 'r_input_p': 22.0, 'sersic_n_input_p': 3.0,
     'Re_input_s': 0.5, 'r_input_s': 23.5, 'sersic_n_input_s': 1.5, 'distance': 1.5},
    {'Re_input_p': 0.3, 'r_input_p': 25.0, 'sersic_n_input_p': 0.8,
     'Re_input_s': 0.4, 'r_input_s': 24.0, 'sersic_n_input_s': 1.0, 'distance': 5.0},
])
print('Custom pair DataFrame:')
print(custom_pairs)

# Run all three emulators
out_blend = predictor.predict_on_pairs(custom_pairs, task='response')
out_self  = predictor.predict_on_pairs(custom_pairs, task='self_response')
out_det   = predictor.predict_on_pairs(custom_pairs, task='detection')

print('\nPredictions:')
print(pd.DataFrame({
    'blending_response': out_blend['response'],
    'self_response':     out_self['self_response'],
    'detection_prob':    out_det['detection_prob'],
}))

### Example C: If you already have a "reg_cat" from the pipeline

If you've already built a pair catalogue using the full pipeline (e.g. `blendemu.response.retrieve_response`, or reusing a `reg_fea` from `icat2reg`), you can feed it straight in — the same raw-feature column names apply:

In [ ]:
# Reuse the `reg_features` DataFrame we built earlier (from section 5)
# It already contains Re_input_p, r_input_p, etc. — no need to re-run the pairing
print(f'Existing reg_features has {reg_features.shape[0]:,} pairs')
print(f'Columns include: {[c for c in reg_features.columns if "input" in c or c == "distance"][:8]}...')

# reg_features already has the *_scaled columns from the earlier prediction call,
# so we can use rescaled=True to skip the rescaling step (small speedup)
out = predictor.predict_on_pairs(reg_features, task='self_response', rescaled=True)
print(f"\nSelf-response added: mean = {out['self_response'].mean():.4f}")